# TerraShift &mdash; Training Notebook (Colab GPU)

Reproduces the two TerraShift baselines: an RGB ResNet50 and a 13-band multispectral ResNet50,
both transfer-learned on EuroSAT Sentinel-2 imagery.

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better). Each training run is
10 epochs, frozen backbone, only the head (and, for multispectral, the expanded first conv)
trained -- this should take well under an hour per model on a T4.

This notebook is self-contained (no dependency on the TerraShift repo being cloned). Optionally
mount Google Drive at the end to persist checkpoints and metrics beyond the Colab session.

In [ ]:
!pip install -q rasterio grad-cam

In [ ]:
import json, random, time, zipfile, urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report,
)
from tqdm.auto import tqdm

CLASSES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]
SEED = 42
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected -- Runtime > Change runtime type > GPU, then re-run this cell.")

def split_indices(n, seed=SEED):
    rng = random.Random(seed)
    idx = list(range(n))
    rng.shuffle(idx)
    n_train = int(round(n * TRAIN_FRAC))
    n_val = int(round(n * VAL_FRAC))
    return idx[:n_train], idx[n_train:n_train + n_val], idx[n_train + n_val:]

## Part 1 -- RGB baseline

In [ ]:
rgb_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

rgb_full = datasets.EuroSAT(root="./data", download=True, transform=rgb_transform)
assert list(rgb_full.classes) == CLASSES, rgb_full.classes

train_idx, val_idx, test_idx = split_indices(len(rgb_full))
rgb_train, rgb_val, rgb_test = Subset(rgb_full, train_idx), Subset(rgb_full, val_idx), Subset(rgb_full, test_idx)
print(f"RGB split -> train={len(rgb_train)} val={len(rgb_val)} test={len(rgb_test)}")

BATCH_SIZE = 32
rgb_train_loader = DataLoader(rgb_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
rgb_val_loader = DataLoader(rgb_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
rgb_test_loader = DataLoader(rgb_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
def build_rgb_resnet50(num_classes=10):
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

rgb_model = build_rgb_resnet50().to(device)
total, trainable = count_params(rgb_model)
print(f"RGB model: {total:,} total / {trainable:,} trainable params")

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train: bool):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    torch.set_grad_enabled(train)
    for inputs, labels in tqdm(loader, leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        if train:
            optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    torch.set_grad_enabled(True)
    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, device, epochs=10, lr=0.001, run_name="run"):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, train_loader, criterion, optimizer, device, True)
        vl, va = run_epoch(model, val_loader, criterion, optimizer, device, False)
        history["train_loss"].append(tl); history["train_acc"].append(ta)
        history["val_loss"].append(vl); history["val_acc"].append(va)
        print(f"[{run_name}] epoch {epoch}/{epochs} train_loss={tl:.4f} train_acc={ta:.4f} "
              f"val_loss={vl:.4f} val_acc={va:.4f} ({time.time()-t0:.1f}s)")
        if va > best_val_acc:
            best_val_acc = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
        torch.save(best_state, f"{run_name}_best.pth")
    return history, best_val_acc

In [ ]:
rgb_history, rgb_best_val_acc = train_model(
    rgb_model, rgb_train_loader, rgb_val_loader, device, epochs=10, lr=0.001, run_name="rgb")
print(f"RGB best val acc: {rgb_best_val_acc:.4f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(rgb_history["train_loss"], label="train"); plt.plot(rgb_history["val_loss"], label="val")
plt.title("RGB loss"); plt.xlabel("epoch"); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(rgb_history["train_acc"], label="train"); plt.plot(rgb_history["val_acc"], label="val")
plt.title("RGB accuracy"); plt.xlabel("epoch"); plt.legend()
plt.tight_layout(); plt.savefig("rgb_curves.png", dpi=150); plt.show()

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    labels_all, preds_all = [], []
    for inputs, labels in loader:
        outputs = model(inputs.to(device))
        preds_all.extend(outputs.argmax(1).cpu().tolist())
        labels_all.extend(labels.tolist())
    labels_all, preds_all = np.array(labels_all), np.array(preds_all)
    acc = accuracy_score(labels_all, preds_all)
    p, r, f1, _ = precision_recall_fscore_support(labels_all, preds_all, average="weighted", zero_division=0)
    cm = confusion_matrix(labels_all, preds_all)
    report = classification_report(labels_all, preds_all, target_names=CLASSES, output_dict=True, zero_division=0)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1, "confusion_matrix": cm, "report": report}

rgb_metrics = evaluate(rgb_model, rgb_test_loader, device)
print(f"RGB test: accuracy={rgb_metrics['accuracy']:.4f} precision={rgb_metrics['precision']:.4f} "
      f"recall={rgb_metrics['recall']:.4f} f1={rgb_metrics['f1']:.4f}")

plt.figure(figsize=(8, 6))
sns.heatmap(rgb_metrics["confusion_matrix"], annot=True, fmt="d", cmap="viridis",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("RGB confusion matrix (test)")
plt.tight_layout(); plt.savefig("rgb_confusion_matrix.png", dpi=150); plt.show()

with open("rgb_test_metrics.json", "w") as f:
    json.dump({k: v for k, v in rgb_metrics.items() if k != "confusion_matrix"} |
              {"confusion_matrix": rgb_metrics["confusion_matrix"].tolist()}, f, indent=2)

## Part 2 -- Multispectral (13-band) baseline

In [ ]:
MS_ZIP_URL = "https://madm.dfki.de/files/sentinel/EuroSATallBands.zip"
ms_root = Path("./data_ms")
ms_root.mkdir(exist_ok=True)
tif_root = ms_root / "EuroSATallBands" / "ds" / "images" / "remote_sensing" / "otherDatasets" / "sentinel_2" / "tif"

if not tif_root.exists():
    zip_path = ms_root / "EuroSATallBands.zip"
    if not zip_path.exists():
        print("Downloading EuroSATallBands.zip (large, may take several minutes)...")
        urllib.request.urlretrieve(MS_ZIP_URL, zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(ms_root / "EuroSATallBands")

import rasterio

class EuroSATMultispectralDataset(Dataset):
    def __init__(self, tif_root):
        self.samples = []
        for cls_idx, cls in enumerate(CLASSES):
            for f in sorted((Path(tif_root) / cls).glob("*.tif")):
                self.samples.append((str(f), cls_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        with rasterio.open(path) as src:
            arr = src.read().astype(np.float32)
        arr = arr / 10000.0
        return torch.from_numpy(arr), label

ms_full = EuroSATMultispectralDataset(tif_root)
ms_train_idx, ms_val_idx, ms_test_idx = split_indices(len(ms_full))
ms_train = Subset(ms_full, ms_train_idx)
ms_val = Subset(ms_full, ms_val_idx)
ms_test = Subset(ms_full, ms_test_idx)
print(f"Multispectral split -> train={len(ms_train)} val={len(ms_val)} test={len(ms_test)}")

ms_train_loader = DataLoader(ms_train, batch_size=32, shuffle=True, num_workers=2)
ms_val_loader = DataLoader(ms_val, batch_size=32, shuffle=False, num_workers=2)
ms_test_loader = DataLoader(ms_test, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
def build_multispectral_resnet50(num_classes=10, in_channels=13):
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    old_conv1 = model.conv1
    rgb_weight = old_conv1.weight.data.clone()
    new_conv1 = nn.Conv2d(in_channels, old_conv1.out_channels, kernel_size=old_conv1.kernel_size,
                           stride=old_conv1.stride, padding=old_conv1.padding,
                           bias=old_conv1.bias is not None)
    new_weight = torch.zeros(old_conv1.out_channels, in_channels, *old_conv1.kernel_size)
    new_weight[:, :3, :, :] = rgb_weight
    avg_weight = rgb_weight.mean(dim=1, keepdim=True)
    new_weight[:, 3:, :, :] = avg_weight.repeat(1, in_channels - 3, 1, 1)
    new_conv1.weight.data = new_weight
    model.conv1 = new_conv1
    for p in model.parameters():
        p.requires_grad = False
    for p in model.conv1.parameters():
        p.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

ms_model = build_multispectral_resnet50().to(device)
total, trainable = count_params(ms_model)
print(f"Multispectral model: {total:,} total / {trainable:,} trainable params")

In [ ]:
ms_history, ms_best_val_acc = train_model(
    ms_model, ms_train_loader, ms_val_loader, device, epochs=10, lr=0.001, run_name="multispectral")
print(f"Multispectral best val acc: {ms_best_val_acc:.4f}")

ms_metrics = evaluate(ms_model, ms_test_loader, device)
print(f"Multispectral test: accuracy={ms_metrics['accuracy']:.4f} precision={ms_metrics['precision']:.4f} "
      f"recall={ms_metrics['recall']:.4f} f1={ms_metrics['f1']:.4f}")

plt.figure(figsize=(8, 6))
sns.heatmap(ms_metrics["confusion_matrix"], annot=True, fmt="d", cmap="viridis",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Multispectral confusion matrix (test)")
plt.tight_layout(); plt.savefig("multispectral_confusion_matrix.png", dpi=150); plt.show()

with open("multispectral_test_metrics.json", "w") as f:
    json.dump({k: v for k, v in ms_metrics.items() if k != "confusion_matrix"} |
              {"confusion_matrix": ms_metrics["confusion_matrix"].tolist()}, f, indent=2)

## Reading the comparison

If the RGB run outperforms the multispectral run here, report it as: *"under this experimental
configuration, the RGB baseline outperformed the multispectral baseline"* -- not as a general claim
that RGB is better than multispectral imagery. The reference numbers from the original documented
run were ~93.48% RGB / ~85.93% multispectral test accuracy; treat whatever this notebook actually
produces as the numbers to publish.

## Part 3 -- Grad-CAM (explainability)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

def denormalize_rgb(tensor):
    mean = np.array(IMAGENET_MEAN); std = np.array(IMAGENET_STD)
    img = tensor.permute(1, 2, 0).cpu().numpy()
    return np.clip(std * img + mean, 0, 1)

def gradcam_on_index(model, dataset, index, mode, tag=""):
    model.eval()
    input_tensor, true_label = dataset[index]
    input_batch = input_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(input_batch), dim=1)[0]
    pred_label = int(probs.argmax()); confidence = float(probs[pred_label])

    cam = GradCAM(model=model, target_layers=[model.layer4[-1]])
    grayscale_cam = cam(input_tensor=input_batch)[0]

    if mode == "rgb":
        rgb_img = denormalize_rgb(input_tensor)
    else:
        arr = input_tensor[[3, 2, 1], :, :].cpu().numpy()
        rgb_img = np.clip(np.transpose(arr, (1, 2, 0)) * 3.0, 0, 1)

    overlay = show_cam_on_image(rgb_img.astype(np.float32), grayscale_cam, use_rgb=True)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(rgb_img); axes[0].set_title(f"True: {CLASSES[true_label]}"); axes[0].axis("off")
    axes[1].imshow(overlay); axes[1].set_title(f"Pred: {CLASSES[pred_label]} ({confidence:.1%})"); axes[1].axis("off")
    plt.tight_layout()
    fname = f"gradcam_{mode}_{tag or 'idx'+str(index)}.png"
    plt.savefig(fname, dpi=150); plt.show()
    correct = "correct" if pred_label == true_label else "MISCLASSIFIED"
    print(f"[{correct}] true={CLASSES[true_label]} pred={CLASSES[pred_label]} confidence={confidence:.4f} -> {fname}")

# Example: a correctly classified RGB test example (pick any index)
gradcam_on_index(rgb_model, rgb_test, index=0, mode="rgb", tag="example")

# To reproduce the documented multispectral error-analysis example (Residential -> SeaLake, ~95%
# confidence), scan the multispectral test set for a high-confidence Residential misclassification:
@torch.no_grad()
def find_overconfident_misclassification(model, dataset, true_class_name, device, min_confidence=0.9):
    model.eval()
    true_idx = CLASSES.index(true_class_name)
    for i in range(len(dataset)):
        x, y = dataset[i]
        if y != true_idx:
            continue
        probs = F.softmax(model(x.unsqueeze(0).to(device)), dim=1)[0]
        pred = int(probs.argmax())
        if pred != y and float(probs[pred]) >= min_confidence:
            return i
    return None

idx = find_overconfident_misclassification(ms_model, ms_test, "Residential", device, min_confidence=0.9)
if idx is not None:
    gradcam_on_index(ms_model, ms_test, index=idx, mode="multispectral", tag="error_residential")
else:
    print("No overconfident Residential misclassification found in this run's test set.")

## Save checkpoints and results (optional: Google Drive)

Colab sessions are ephemeral -- copy `rgb_best.pth`, `multispectral_best.pth`, the `*_test_metrics.json`
files, and the PNG figures out before the runtime recycles. Uncomment to mount Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil, glob
# dest = "/content/drive/MyDrive/TerraShift/research_outputs"
# import os; os.makedirs(dest, exist_ok=True)
# for f in glob.glob("*.pth") + glob.glob("*.json") + glob.glob("*.png"):
#     shutil.copy(f, dest)
# print("Copied outputs to", dest)